In [1]:
# =============================================================================
# pulp_pmedian.py — Full p-median optimization using PuLP + CBC solver
#
# Save this file to: D:\GIS_Seminar_Project\PuLP_Outputs\
#
# WHY PULP: PuLP uses the free CBC solver — no license, no variable limit.
# This solves the FULL problem with all 733 demand nodes × 25 candidates,
# giving the unconstrained exact optimum for comparison against CPLEX.
#
# INSTALL (run once in your GIS_ML environment):
#   pip install pulp
#
# INPUTS (from Colab_Inputs — no need to copy, just update paths below):
#   demand_nodes.csv
#   distance_matrix_network.csv
#
# OUTPUTS (saved to PuLP_Outputs\):
#   pulp_results.json
#   pulp_assignments_p5.csv
#   pulp_assignments_p8.csv
#   pulp_assignments_p10.csv
#
# These four files go into Colab_Final\ for Phase 2B.
# =============================================================================

import pandas as pd
import numpy as np
import pulp
import json, os, time

# ── PATHS ────────────────────────────────────────────────────────────────────
DEMAND_CSV = r"D:\GIS_Seminar_Project\Colab_Inputs\demand_nodes.csv"
MATRIX_CSV = r"D:\GIS_Seminar_Project\Colab_Inputs\distance_matrix_network.csv"
OUTPUT_DIR = r"D:\GIS_Seminar_Project\PuLP_Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

P_VALUES = [5, 8, 10]

# ── 25 candidate shelters ────────────────────────────────────────────────────
candidates_data = [
    {"SID": 0,  "NAME": "Clearwater Fundamental Middle",       "LAT": 27.976248, "LON": -82.766487},
    {"SID": 1,  "NAME": "Palm Harbor Middle",                  "LAT": 28.066762, "LON": -82.752829},
    {"SID": 2,  "NAME": "John Hopkins Middle",                 "LAT": 27.763104, "LON": -82.656309},
    {"SID": 3,  "NAME": "Sexton Elementary",                   "LAT": 27.822357, "LON": -82.659844},
    {"SID": 4,  "NAME": "Gibbs High",                          "LAT": 27.761577, "LON": -82.678299},
    {"SID": 5,  "NAME": "McMullen-Booth Elementary",           "LAT": 27.995527, "LON": -82.712017},
    {"SID": 6,  "NAME": "Carwise Middle",                      "LAT": 28.088628, "LON": -82.726590},
    {"SID": 7,  "NAME": "Jamerson Elementary",                 "LAT": 27.758239, "LON": -82.682283},
    {"SID": 8,  "NAME": "Sanderlin K-8",                       "LAT": 27.747297, "LON": -82.665967},
    {"SID": 9,  "NAME": "Ross Norton",                         "LAT": 27.945113, "LON": -82.793368},
    {"SID": 10, "NAME": "Campbell Park Elementary",            "LAT": 27.763674, "LON": -82.649600},
    {"SID": 11, "NAME": "New Heights Elementary",              "LAT": 27.807558, "LON": -82.682420},
    {"SID": 12, "NAME": "Fairmount Park Elementary",           "LAT": 27.764914, "LON": -82.690303},
    {"SID": 13, "NAME": "Belleair Elementary",                 "LAT": 27.950563, "LON": -82.788892},
    {"SID": 14, "NAME": "Skycrest Elementary",                 "LAT": 27.966234, "LON": -82.761041},
    {"SID": 15, "NAME": "Largo High",                          "LAT": 27.919742, "LON": -82.786281},
    {"SID": 16, "NAME": "Lealman Exchange",                    "LAT": 27.818585, "LON": -82.693162},
    {"SID": 17, "NAME": "Mildred Helms Elementary",            "LAT": 27.912518, "LON": -82.797080},
    {"SID": 18, "NAME": "Melrose Elementary",                  "LAT": 27.757180, "LON": -82.657401},
    {"SID": 19, "NAME": "Palm Harbor University High",         "LAT": 28.085000, "LON": -82.760587},
    {"SID": 20, "NAME": "Palm Harbor University High Bldg 19", "LAT": 28.084940, "LON": -82.760508},
    {"SID": 21, "NAME": "Clearwater High",                     "LAT": 27.959558, "LON": -82.756357},
    {"SID": 22, "NAME": "Palm Harbor CSA",                     "LAT": 28.081885, "LON": -82.757176},
    {"SID": 23, "NAME": "The Coliseum",                        "LAT": 27.776730, "LON": -82.641115},
    {"SID": 24, "NAME": "White Chapel",                        "LAT": 28.076766, "LON": -82.765247},
]
candidates = pd.DataFrame(candidates_data)

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
demand = pd.read_csv(DEMAND_CSV, dtype={"GEOID_JOIN": str})
demand["GEOID_JOIN"] = demand["GEOID_JOIN"].str.replace(r"\.0$", "", regex=True)

matrix = pd.read_csv(MATRIX_CSV, index_col=0, dtype={0: str})
matrix.index = matrix.index.astype(str).str.replace(r"\.0$", "", regex=True)
D = matrix.values.astype(float)
W = demand["WEIGHTED_DEMAND"].values

n_I, n_J = len(W), D.shape[1]
print(f"Demand nodes:    {n_I}  (full, no aggregation)")
print(f"Candidates:      {n_J}")
print(f"Total variables: {n_J + n_I*n_J:,}  (no solver limit with PuLP/CBC)")
print(f"Solving for p =  {P_VALUES}")
print()

# ── p-MEDIAN SOLVER (PuLP/CBC) ────────────────────────────────────────────────
def solve_pmedian_pulp(W, D, p, time_limit=600):
    """
    Solve the p-median problem using PuLP with the CBC solver.

    Variables:
      x[j]    = 1 if shelter j is open
      y[i][j] = 1 if demand node i assigned to shelter j

    Objective:
      Minimize sum_i sum_j  W[i] * D[i][j] * y[i][j]

    Constraints:
      sum_j x[j] = p              (exactly p shelters open)
      sum_j y[i][j] = 1           (each demand node assigned once)
      y[i][j] <= x[j]             (only assign to open shelters)
    """
    n_I, n_J = len(W), D.shape[1]

    prob = pulp.LpProblem(f"pmedian_p{p}", pulp.LpMinimize)

    # Decision variables
    x = [pulp.LpVariable(f"x_{j}", cat="Binary") for j in range(n_J)]
    y = [[pulp.LpVariable(f"y_{i}_{j}", cat="Binary")
          for j in range(n_J)] for i in range(n_I)]

    # Objective: minimise total flood-risk-weighted travel time
    prob += pulp.lpSum(
        W[i] * D[i][j] * y[i][j]
        for i in range(n_I) for j in range(n_J)
    )

    # Constraints
    prob += pulp.lpSum(x) == p                                    # exactly p open
    for i in range(n_I):
        prob += pulp.lpSum(y[i]) == 1                             # one assignment
    for i in range(n_I):
        for j in range(n_J):
            prob += y[i][j] <= x[j]                               # open only

    # Solve with CBC — msg=1 shows progress, timeLimit in seconds
    t0     = time.time()
    solver = pulp.PULP_CBC_CMD(msg=1, timeLimit=time_limit)
    prob.solve(solver)
    elapsed = time.time() - t0

    status = pulp.LpStatus[prob.status]
    if status not in ("Optimal", "Not Solved"):
        print(f"  Warning: solver status = {status}")

    selected    = [j for j in range(n_J) if pulp.value(x[j]) > 0.5]
    assignments = [
        next(j for j in range(n_J) if pulp.value(y[i][j]) > 0.5)
        for i in range(n_I)
    ]
    objective = pulp.value(prob.objective)

    return {
        "p":           p,
        "objective":   objective,
        "solve_time":  elapsed,
        "status":      status,
        "selected":    selected,
        "assignments": assignments,
    }

# ── RUN FOR p = 5, 8, 10 ─────────────────────────────────────────────────────
pulp_results = {}
print("=" * 65)
print("PuLP/CBC p-MEDIAN OPTIMIZATION  (full 733 demand nodes)")
print("=" * 65)

for p in P_VALUES:
    print(f"\nSolving p = {p}  (this may take a few minutes)...")
    res = solve_pmedian_pulp(W, D, p)
    pulp_results[p] = res

    print(f"\n  Status:      {res['status']}")
    print(f"  Objective:   {res['objective']:,.2f} person·min")
    print(f"  Solve time:  {res['solve_time']:.1f} s")
    print(f"  Selected shelters ({p}):")
    for j in res["selected"]:
        print(f"    ✓ S{j}: {candidates.iloc[j]['NAME']}")

# ── SAVE OUTPUTS ──────────────────────────────────────────────────────────────
# 1. Summary JSON
summary = {
    str(p): {
        "objective":   res["objective"],
        "solve_time":  res["solve_time"],
        "status":      res["status"],
        "selected":    res["selected"],
    }
    for p, res in pulp_results.items()
}
json_path = os.path.join(OUTPUT_DIR, "pulp_results.json")
with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved: pulp_results.json")

# 2. Assignment CSVs
cols = ["GEOID_JOIN", "LAT", "LON", "POPULATION", "WEIGHTED_DEMAND"]
for col in ["TIER_LABEL", "MULTIPLIER"]:
    if col in demand.columns:
        cols.append(col)

for p, res in pulp_results.items():
    out = demand[cols].copy()
    out["METHOD"]          = "PuLP_CBC"
    out["ASSIGNED_SID"]    = res["assignments"]
    out["ASSIGNED_NAME"]   = [candidates.iloc[j]["NAME"] for j in res["assignments"]]
    out["TRAVEL_TIME_MIN"] = [D[i, res["assignments"][i]] for i in range(n_I)]
    path = os.path.join(OUTPUT_DIR, f"pulp_assignments_p{p}.csv")
    out.to_csv(path, index=False)
    print(f"Saved: pulp_assignments_p{p}.csv")

# ── COMPARISON SUMMARY ────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("PULP RESULTS SUMMARY")
print("=" * 65)
print(f"  {'p':>4}  {'Objective':>16}  {'Time(s)':>9}  {'Status'}")
print(f"  {'-'*55}")
for p, res in pulp_results.items():
    print(f"  {p:>4}  {res['objective']:>16,.2f}  "
          f"{res['solve_time']:>9.1f}  {res['status']}")

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print("\nCopy these four files to Colab_Final\\:")
print("  pulp_results.json")
print("  pulp_assignments_p5.csv")
print("  pulp_assignments_p8.csv")
print("  pulp_assignments_p10.csv")

Demand nodes:    733  (full, no aggregation)
Candidates:      25
Total variables: 18,350  (no solver limit with PuLP/CBC)
Solving for p =  [5, 8, 10]

PuLP/CBC p-MEDIAN OPTIMIZATION  (full 733 demand nodes)

Solving p = 5  (this may take a few minutes)...

  Status:      Optimal
  Objective:   4,184,071.38 person·min
  Solve time:  7.4 s
  Selected shelters (5):
    ✓ S1: Palm Harbor Middle
    ✓ S8: Sanderlin K-8
    ✓ S14: Skycrest Elementary
    ✓ S16: Lealman Exchange
    ✓ S17: Mildred Helms Elementary

Solving p = 8  (this may take a few minutes)...

  Status:      Optimal
  Objective:   3,774,238.41 person·min
  Solve time:  28.2 s
  Selected shelters (8):
    ✓ S0: Clearwater Fundamental Middle
    ✓ S1: Palm Harbor Middle
    ✓ S3: Sexton Elementary
    ✓ S5: McMullen-Booth Elementary
    ✓ S8: Sanderlin K-8
    ✓ S12: Fairmount Park Elementary
    ✓ S16: Lealman Exchange
    ✓ S17: Mildred Helms Elementary

Solving p = 10  (this may take a few minutes)...

  Status:      Opti